# Issue #16 — Failure Severity Diagnostics (NFCorpus)

**Research question:** As a failure affects a larger fraction of the system, does retrieval nDCG@10 fall more? Do intended harmless controls stay near zero damage?

- Source: the benchmark's `ResultRow` parquet, **not** retrieval model output or fabricated measurements.
- Only **NFCorpus** is analyzed here; SciFact and FiQA remain held out.
- Multiple detectors create duplicate labels for the *same* query window. The analysis first reduces these to one row per `(run_id, failure, severity, seed, trial)`.
- Confidence intervals resample **whole independent seeds**, not overlapping windows/trials. One seed gives a point estimate **without a CI**; even multiple seeds share the same fixed NFCorpus query population.
- A negative control is **not proven harmless** merely because a significance test fails to reject a drop. Define an equivalence margin with the team before claiming it is flat.

**To use:** run a real NFCorpus benchmark once the F1–F5/negative-control injectors are integrated, then set `RESULT_PATH` below. The notebook can already be opened, but does not claim any results before the parquet exists.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from silent_rag.eval.severity_analysis import (
    prepare_windows, severity_summary, adjacent_severity_trends, plot_severity_curves
)

# Change this if your NFCorpus run writes to a different parquet.
RESULT_PATH = Path("results/dev.parquet")
FIGURE_DIR = Path("results/severity_analysis/figures")
N_BOOT = 2000
BOOTSTRAP_SEED = 20261029


In [ ]:
if not RESULT_PATH.exists():
    print(f"No real benchmark results yet: {RESULT_PATH}")
    print("This is expected until the injectors are merged and a NFCorpus run finishes.")
    windows = None
else:
    raw = pd.read_parquet(RESULT_PATH)
    windows = prepare_windows(raw)
    print(f"Loaded {len(raw)} ResultRows -> {len(windows)} unique experiment windows")
    display(windows.groupby(["failure", "severity"], sort=True).agg(
        seeds=("seed", "nunique"), windows=("trial", "size")
    ).reset_index())

In [ ]:
if windows is not None:
    summary = severity_summary(windows, n_boot=N_BOOT, seed=BOOTSTRAP_SEED)
    display(summary)
    paths = plot_severity_curves(summary, FIGURE_DIR)
    print("Figures:")
    for path in paths:
        print(" ", path)
    # Save analysis tables alongside the figures; no held-out evaluation.
    FIGURE_DIR.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(FIGURE_DIR.parent / "severity_summary.csv", index=False)

In [ ]:
if windows is not None:
    trends = adjacent_severity_trends(windows, n_boot=N_BOOT, seed=BOOTSTRAP_SEED)
    display(trends)
    trends.to_csv(FIGURE_DIR.parent / "adjacent_severity_trends.csv", index=False)
    print("'mean_decreased' signals an observed violation of monotonicity, not necessarily a statistically certain one.")
    print("For controls, examine mean damage and CI; no equivalence margin has been selected yet.")

## Interpretation / follow-up checklist

- For **harmful failures**, review both the average drop and adjacent-severity paired differences. Report any consistent decreases to the injector owner with the specific seed/trial examples; do not modify their injector without coordination.
- For **negative controls**, report whether the injected change shifts detector scores (separate experiment) and whether retrieval damage stays within a **team-agreed** practically harmless margin. Non-significance alone cannot certify a control.
- If confidence intervals are missing, rerun the benchmark with `seeds: [0, 1, 2, ...]` instead of a single `seed:` **after** the real injector/config is available. Multiple seed blocks vary reference windows and damage randomization, but are still conditional on the fixed NFCorpus benchmark.
- **Do not** run the held-out final config to decide thresholds, severity definitions or analysis choices.
